# 🏥 Fed-ISIC2019 연합학습 실험 노트북 (Colab Phase-by-Phase Runner)

팀원들이 Google Colab에서 **2단계 전략(탐색 시드로 μ 선택 → 최종 3시드 검증)** 으로 실험을 수행하고, 결과(`Mean ± Std` 요약표 및 표준편차 음영 그래프)를 구글 드라이브에 저장하도록 구성된 노트북입니다.

### 📋 실험 구성 및 전략
- **모델**: `EfficientNet-B0` (ImageNet Pretrained, `BatchNorm` 유지, 8-Class Classifier)
- **데이터 분할 모드 (`--mode`)**:
  - `top3`: 상위 3개 주요 센터 (센터 0, 1, 2) — FLamby train 15,784장 / test 3,946장
  - `merge6`: 전체 6개 센터(0~5)를 3개 클라이언트로 병합 `[(0,3), (1,4), (2,5)]` — FLamby train 18,597장 / test 4,650장
- **검증(validation) 분할**: 각 센터의 FLamby train에서 **10%를 층화로 떼어 validation**으로 사용합니다(`--val_fraction 0.1`, 분할 시드 `--split_seed 0` 고정). 나머지 90%만 학습합니다.
- **2단계 시드 실험 전략**:
  - **Stage 1 (`seed=41`, 탐색 전용)**: `FedAvg`(μ=0 기준선)와 `FedProx`의 $\mu$ 후보를 **validation** 기준으로 비교해 최적 $\mu^*$ 선정. 이 시드는 최종 결과에 포함하지 않습니다.
  - **Stage 2 (`seed=42, 100, 2026`)**: 최종 설정(`Centralized`, `FedAvg`, `FedProx(BEST_MU)`)을 3개 시드로 실행하고, 학습이 끝나면 **test를 1회만** 평가합니다(`--final_test`).
- **핵심 평가지표**: `balanced_acc`, `macro_f1`(정답에 존재하는 클래스 기준), `macro_f1_gap`(클라이언트 간 최고-최저 Macro-F1), `ece`(15-bin), `weight_divergence`, `cumulative_comm_mb`. 클라이언트별 F1(`macro_f1_client0~2`)도 CSV에 저장됩니다.

> ⚠️ **평가 원칙**: 매 라운드 CSV에 기록되는 모든 지표(`eval_split=val`)는 **validation** 기준이며, 최고 라운드 선택과 $\mu$ 선택에도 validation만 씁니다. FLamby **test는 최종 모델에 대해 1회만** 평가해 `*_final_test.json`에 저장합니다(마지막 라운드 모델 = 주 결과, best-val 모델 = 참고). 이미 결과 파일이 있으면 test를 다시 열지 않습니다.

> 🔁 **세션이 끊겼을 때**: Phase 0 ~ Phase 2 셀을 다시 실행한 뒤, 중단된 실험 셀을 **그대로 다시 실행**하면 됩니다. 모든 실행 명령에 `--resume`이 들어 있어 마지막으로 완료된 라운드 다음부터 이어서 학습합니다.


---
## Phase 0. 코랩 GPU 확인, 구글 드라이브 마운트 및 경로 설정
- `SAVE_DIR`(Drive): 매 라운드 CSV, 최고 성능 `best_*.pt`, resume용 `last_*.pt` 저장
- `DRIVE_DATA_TAR`(Drive): 전처리된 ISIC 이미지 묶음(최초 1회 생성 후 재사용)
- `DATA_PATH`(로컬 `/content`): 학습 시 실제로 읽는 데이터 경로 (Drive에서 직접 읽으면 I/O가 매우 느리므로 매 세션 로컬로 압축 해제)

In [ ]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# Google Drive 마운트 (코랩 환경에서 실행)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("로컬 환경에서 실행 중입니다.")

# 프로젝트 / 결과 / 데이터 경로 설정
PROJECT_DIR = "/content/CDII" if IN_COLAB else os.getcwd()
SAVE_DIR = "/content/drive/MyDrive/CDII_results" if IN_COLAB else "./results"
FLAMBY_DIR = "/content/FLamby" if IN_COLAB else os.path.abspath("./FLamby")
DRIVE_DATA_TAR = "/content/drive/MyDrive/CDII_data/isic2019_preprocessed.tar" if IN_COLAB else None
DATA_PATH = "/content/isic_data" if IN_COLAB else os.path.abspath("./isic_data")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)
print(f"[설정 완료] SAVE_DIR={SAVE_DIR}")
print(f"[설정 완료] DATA_PATH={DATA_PATH} | DRIVE_DATA_TAR={DRIVE_DATA_TAR}")

---
## Phase 1. 프로젝트 코드 준비, FLamby 설치, 데이터 준비 및 데이터로더 검증

In [ ]:
# 1-1. 프로젝트 코드 준비: Git 레포지토리에서 클론하거나 이미 업로드된 프로젝트 폴더로 이동
# 수정 브랜치(validation 분할 포함)를 받습니다. main에 합쳐진 뒤에는 REPO_BRANCH = "main"으로 바꾸세요.
REPO_URL = "https://github.com/NOWEAZ/Capstone-Design-II.git"
REPO_BRANCH = "fix/validation-split-and-metrics"
if IN_COLAB and not os.path.exists(PROJECT_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}

if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}
else:
    raise FileNotFoundError(f"{PROJECT_DIR} 가 없습니다. 위 clone이 실패했거나 프로젝트 폴더를 업로드하지 않았습니다.")
print("현재 작업 디렉토리:", os.getcwd())
!git log --oneline -1
!ls -la


In [ ]:
# 1-2. 필수 패키지 및 FLamby 설치 (FLamby는 PyPI 패키지가 아니므로 공식 방식대로 clone 후 editable 설치)
import sys

!pip install -q -r requirements.txt
if not os.path.exists(FLAMBY_DIR):
    !git clone https://github.com/owkin/FLamby.git {FLAMBY_DIR}
!cd {FLAMBY_DIR} && pip install -q -e ".[isic2019]"

# editable 설치는 커널 재시작 전까지 현재 노트북 커널에서 import되지 않을 수 있어 경로를 직접 추가
# (main.py는 !python 으로 새 프로세스에서 실행되므로 영향 없음)
if FLAMBY_DIR not in sys.path:
    sys.path.insert(0, FLAMBY_DIR)
import flamby
print("FLamby import 성공:", flamby.__file__)

In [ ]:
# 1-3. Fed-ISIC2019 데이터 준비
#  - 매 세션: Drive의 전처리 tar를 로컬(/content)로 압축 해제 (수 분 소요)
#  - 최초 1회만: ISIC2019 원본 다운로드 -> FLamby 전처리(resize) -> 전처리 결과를 Drive에 tar로 보관
#  ※ 최초 다운로드 전 ISIC 2019 / HAM10000 데이터 라이선스(CC-BY-NC)를 확인하세요.
#    다운로드 스크립트가 동의 여부를 물으면 셀 입력창에 응답해야 합니다.
PREPROCESSED_NAME = "ISIC_2019_Training_Input_preprocessed"
PREPROCESSED_DIR = os.path.join(DATA_PATH, PREPROCESSED_NAME)

if os.path.isdir(PREPROCESSED_DIR) and len(os.listdir(PREPROCESSED_DIR)) > 0:
    print(f"[데이터] 로컬 전처리 데이터가 이미 존재합니다: {PREPROCESSED_DIR}")
elif DRIVE_DATA_TAR is not None and os.path.exists(DRIVE_DATA_TAR):
    print("[데이터] Drive의 전처리 tar를 로컬로 압축 해제합니다...")
    !tar -xf "{DRIVE_DATA_TAR}" -C "{DATA_PATH}"
else:
    print("[데이터] 최초 실행: ISIC2019 원본 다운로드 및 전처리를 시작합니다 (약 9GB, 수십 분 소요).")
    SCRIPT_DIR = os.path.join(FLAMBY_DIR, "flamby", "datasets", "fed_isic2019", "dataset_creation_scripts")
    %cd {SCRIPT_DIR}
    !python download_isic.py --output-folder "{DATA_PATH}"
    !python resize_images.py
    %cd {PROJECT_DIR}
    if DRIVE_DATA_TAR is not None:
        os.makedirs(os.path.dirname(DRIVE_DATA_TAR), exist_ok=True)
        print("[데이터] 다음 세션부터 재사용할 수 있도록 전처리 결과를 Drive에 tar로 저장합니다...")
        !tar -cf "{DRIVE_DATA_TAR}" -C "{DATA_PATH}" {PREPROCESSED_NAME}

n_imgs = len(os.listdir(PREPROCESSED_DIR)) if os.path.isdir(PREPROCESSED_DIR) else 0
print(f"[데이터] 전처리 이미지 수: {n_imgs} (정상: 약 23,247장)")

In [ ]:
# 1-4. 데이터로더 및 역빈도 클래스 가중치 정상 작동 확인 (Smoke Test)
from data_utils import get_dataloaders, compute_class_weights

MODE_TEST = "top3"  # 'top3' 또는 'merge6'
client_loaders, centralized_loader, val_loader, test_loader = get_dataloaders(
    mode=MODE_TEST,
    batch_size=32,
    num_workers=2,
    data_path=DATA_PATH,
    seed=42,
    val_fraction=0.1,
    split_seed=0,
)

print(f"=== Mode: {MODE_TEST} 데이터셋 요약 ===")
for idx, loader in enumerate(client_loaders):
    w = compute_class_weights(loader, num_classes=8)
    print(f"Client {idx} | Train: {len(loader.dataset):5d} | Val: {len(val_loader.client_eval_loaders[idx].dataset):4d} | Test: {len(test_loader.client_eval_loaders[idx].dataset):4d} | 클래스 가중치: {w.numpy().round(2)}")

print(f"Centralized 통합 Train 샘플 수: {len(centralized_loader.dataset)}")
print(f"Global 통합 Val 샘플 수       : {len(val_loader.dataset)}")
print(f"Global 통합 Test 샘플 수      : {len(test_loader.dataset)}")

---
## Phase 2. 공통 하이퍼파라미터 및 시드 설정
- 연합학습(`FedAvg`, `FedProx`): `rounds=20`, `local_epochs=2`
- `Centralized`: `rounds=20`, `local_epochs=1`
- `COMMON_ARGS`에 `--data_path`, `--save_dir`, `--resume`, `--val_fraction`, `--split_seed`가 포함되어 모든 실행 셀에 공통 적용됩니다.
- 탐색 시드(`EXPLORE_SEED`)는 최종 시드(`FINAL_SEEDS`)와 **겹치지 않게** 둡니다. 그래야 $\mu$ 선택에 쓴 실험이 최종 결과에 섞이지 않습니다.


In [ ]:
MODE = "top3"                  # 'top3' 또는 'merge6'
ROUNDS = 20                    # 총 통신 라운드 수
FL_LOCAL_EPOCHS = 2            # 연합학습(FedAvg, FedProx) 라운드당 로컬 에폭 수
CENTRAL_LOCAL_EPOCHS = 1       # Centralized 라운드당 에폭 수
BATCH_SIZE = 32
LR = 1e-4
NUM_WORKERS = 2                # 코랩 기본 런타임 권장값: 2

EXPLORE_SEED = 41              # Stage 1: mu 탐색 전용 시드 (최종 결과에 포함하지 않음)
FINAL_SEEDS = [42, 100, 2026]  # Stage 2: 최종 결과용 시드 3개

VAL_FRACTION = 0.1             # 센터별 train에서 떼어 낼 검증 비율
SPLIT_SEED = 0                 # train/val 분할 시드 (모든 실행에서 동일하게 고정) 추가 시드

COMMON_ARGS = (
    f'--mode {MODE} --rounds {ROUNDS} --batch_size {BATCH_SIZE} --lr {LR} '
    f'--num_workers {NUM_WORKERS} --val_fraction {VAL_FRACTION} --split_seed {SPLIT_SEED} '
    f'--data_path "{DATA_PATH}" --save_dir "{SAVE_DIR}" --resume'
)
print(f"[실험 설정] FL_LOCAL_EPOCHS={FL_LOCAL_EPOCHS}, CENTRAL_LOCAL_EPOCHS={CENTRAL_LOCAL_EPOCHS}")
print(f"[공통 인자] {COMMON_ARGS}")
print(f"[시드 설정] Stage 1(탐색)={EXPLORE_SEED} | Stage 2(최종)={FINAL_SEEDS}")

---
## Phase 3. [Stage 1] 탐색 시드(`seed=41`)로 $\mu$ 선택
`FedAvg`($\mu=0$ 기준선)와 `FedProx`($\mu$ 후보군)를 탐색 시드로 1회씩 실행해 최적의 $\mu^*$를 **validation 기준**으로 찾습니다. 이 시드의 결과는 최종 표에 포함되지 않습니다.


In [ ]:
# 3-1. Centralized (탐색 시드). mu 선택에는 필요하지 않으므로 기본은 건너뜁니다.
#      (Centralized는 Stage 2에서 최종 시드 3개로 실행됩니다. 탐색 단계에서도 보고 싶을 때만 True로 바꾸세요.)
RUN_CENTRALIZED_EXPLORE = False

if RUN_CENTRALIZED_EXPLORE:
    !python main.py --method centralized --local_epochs {CENTRAL_LOCAL_EPOCHS} --seed {EXPLORE_SEED} {COMMON_ARGS}
else:
    print("[건너뜀] Centralized 탐색 실행 (RUN_CENTRALIZED_EXPLORE=False)")


In [ ]:
# 3-2. FedAvg = mu 0 기준선 (탐색 시드, local_epochs=2)
!python main.py --method fedavg --mu 0.0 --local_epochs {FL_LOCAL_EPOCHS} --seed {EXPLORE_SEED} {COMMON_ARGS}


In [ ]:
# 3-3. FedProx mu 후보군 탐색 (탐색 시드, local_epochs=2)
MU_CANDIDATES = [0.001, 0.01, 0.1, 1.0]

for mu_val in MU_CANDIDATES:
    print(f"\n================ [Stage 1] FedProx (mu={mu_val}, seed={EXPLORE_SEED}) ================")
    !python main.py --method fedprox --mu {mu_val} --local_epochs {FL_LOCAL_EPOCHS} --seed {EXPLORE_SEED} {COMMON_ARGS}


In [ ]:
# 3-4. Stage 1 (탐색 시드) 결과 비교 -> 최적 mu 선정
# 모든 지표는 validation 기준입니다. 한 라운드 값은 흔들리므로 마지막 LAST_K 라운드 평균으로 비교합니다.
import glob
import pandas as pd

LAST_K = 3

stage1_rows = []
for path in sorted(glob.glob(os.path.join(SAVE_DIR, f"*_{MODE}_seed{EXPLORE_SEED}_metrics.csv"))):
    df = pd.read_csv(path)
    if df.empty:
        continue
    tail = df.sort_values("round").tail(LAST_K)
    stage1_rows.append({
        "Method": df["method"].iloc[0],
        "Mu": float(df["mu"].iloc[0]),
        "Done Rounds": int(df["round"].max()),
        f"Val Macro F1 (last {LAST_K} mean)": tail["macro_f1"].mean(),
        f"Val Macro F1 SD (last {LAST_K})": tail["macro_f1"].std(ddof=1),
        f"Val Bal Acc (last {LAST_K} mean)": tail["balanced_acc"].mean(),
        f"Val F1 Gap (last {LAST_K} mean)": tail["macro_f1_gap"].mean(),
        f"Val ECE (last {LAST_K} mean)": tail["ece"].mean(),
        f"Weight Div (last {LAST_K} mean)": tail["weight_divergence"].mean(),
    })

stage1_df = pd.DataFrame(stage1_rows)
print(f"=== [Stage 1] Seed {EXPLORE_SEED} 탐색 결과 요약 (validation, 마지막 {LAST_K}라운드 평균) ===")
if stage1_df.empty:
    print("탐색 결과가 아직 없습니다. 3-2, 3-3 셀을 먼저 실행하세요.")
else:
    display(stage1_df.sort_values("Mu").reset_index(drop=True).round(4))

    # 완료된 FedProx 후보 중 Val Macro F1(마지막 K라운드 평균)이 가장 높은 mu를 제안 (자동으로 BEST_MU를 정하지는 않음)
    done = stage1_df[(stage1_df["Method"] == "fedprox") & (stage1_df["Done Rounds"] >= ROUNDS)]
    if not done.empty:
        col = f"Val Macro F1 (last {LAST_K} mean)"
        best = done.loc[done[col].idxmax()]
        print(f"제안: Val Macro F1 최고 mu = {best['Mu']:g} ({best[col]:.4f}). 후보 간 차이가 SD보다 작으면 우연일 수 있으니 weight divergence와 함께 판단하세요.")
        print("weight divergence가 mu가 커져도 줄지 않으면 근접 항이 효과가 없다는 뜻입니다(옵티마이저 영향 등).")


---
## Phase 4. [Stage 2] 최종 설정을 최종 시드 3개(`42, 100, 2026`)로 실행
Stage 1 결과를 보고 `BEST_MU`를 **직접 지정**한 뒤 실행하세요. 각 실행은 학습이 끝나면 `--final_test`로 test를 1회만 평가합니다. Phase 5는 이 `BEST_MU`로 FedProx 결과를 필터링합니다.


In [ ]:
BEST_MU = None  # Stage 1 결과(3-4 셀)를 보고 직접 채우세요. 예: BEST_MU = 0.01
assert BEST_MU is not None, "Stage 1 결과(3-4 셀)를 보고 BEST_MU를 먼저 지정하세요."

FINAL_ARGS = f"{COMMON_ARGS} --final_test"  # 학습 종료 후 test를 1회만 평가해 *_final_test.json으로 저장

for s in FINAL_SEEDS:
    print(f"\n================ [Stage 2] Centralized (seed={s}) ================")
    !python main.py --method centralized --local_epochs {CENTRAL_LOCAL_EPOCHS} --seed {s} {FINAL_ARGS}

    print(f"\n================ [Stage 2] FedAvg (seed={s}) ================")
    !python main.py --method fedavg --mu 0.0 --local_epochs {FL_LOCAL_EPOCHS} --seed {s} {FINAL_ARGS}

    print(f"\n================ [Stage 2] FedProx (mu={BEST_MU}, seed={s}) ================")
    !python main.py --method fedprox --mu {BEST_MU} --local_epochs {FL_LOCAL_EPOCHS} --seed {s} {FINAL_ARGS}


---
## Phase 5. 3시드 통합 결과 분석
1. **Final(마지막 라운드) `Mean ± Std` 요약표 (validation)**
2. **Best(최고 Val Balanced Acc 라운드) `Mean ± Std` 요약표 (validation, 참고용)**
3. **Test `Mean ± Std` 요약표** — `--final_test`로 1회 평가한 결과. **마지막 라운드 모델이 보고서 주 결과**, best-val 모델은 참고
4. **목표 정확도 도달 라운드 및 누적 통신량** — 시드별 + 방법별 집계
5. **라운드별 평균 곡선 + `±1 Std` 음영 그래프** 및 **통신 효율성 그래프** (validation)

최종 표에는 `FINAL_SEEDS`만 포함되며(탐색 시드 제외), Stage 1에서 탐색만 하고 탈락한 FedProx $\mu$ 후보는 `BEST_MU` 기준으로 자동 제외됩니다.


In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

METRICS = ["balanced_acc", "macro_f1", "macro_f1_gap", "ece", "weight_divergence"]
BEST_MU_FILTER = globals().get("BEST_MU", None)  # Phase 4의 BEST_MU (미지정 시 모든 mu 표시)
FINAL_SEEDS_FILTER = globals().get("FINAL_SEEDS", None)  # 탐색 시드 결과는 최종 표에서 제외

all_round_dfs = []
for path in sorted(glob.glob(os.path.join(SAVE_DIR, f"*_{MODE}_seed*_metrics.csv"))):
    df = pd.read_csv(path)
    if df.empty:
        continue
    method = df["method"].iloc[0]
    mu_val = float(df["mu"].iloc[0])
    # Stage 1 탐색 시드 결과는 최종 비교에서 제외
    if FINAL_SEEDS_FILTER is not None and int(df["seed"].iloc[0]) not in FINAL_SEEDS_FILTER:
        continue
    # Stage 1에서 탐색만 하고 탈락한 FedProx mu 후보는 최종 비교에서 제외
    if method == "fedprox" and BEST_MU_FILTER is not None and not np.isclose(mu_val, BEST_MU_FILTER):
        continue
    df["group"] = f"{method} (mu={mu_val:g})" if method == "fedprox" else method
    all_round_dfs.append(df)

print("최종 비교 대상:", [f"{d['group'].iloc[0]} / seed {int(d['seed'].iloc[0])}" for d in all_round_dfs])

records = []
for df in all_round_dfs:
    done = int(df["round"].max())
    if done < ROUNDS:
        print(f"[경고] {df['group'].iloc[0]} seed {int(df['seed'].iloc[0])}: {done}/{ROUNDS} 라운드만 완료 -> 해당 셀을 다시 실행하면 --resume으로 이어서 학습합니다.")
    final_row = df.loc[df["round"].idxmax()]
    best_row = df.loc[df["balanced_acc"].idxmax()]
    rec = {
        "Group": df["group"].iloc[0],
        "Seed": int(df["seed"].iloc[0]),
        "Done Rounds": done,
        "Best Round": int(best_row["round"]),
        "best_comm_mb": best_row["cumulative_comm_mb"],
    }
    for m in METRICS:
        rec[f"final_{m}"] = final_row[m]
        rec[f"best_{m}"] = best_row[m]
    records.append(rec)

rec_df = pd.DataFrame(records)


def mean_std_table(rec_df, prefix):
    rows = []
    for grp, gdf in rec_df.groupby("Group", sort=False):
        row = {"Method": grp, "N_Seeds": len(gdf)}
        for m in METRICS:
            vals = gdf[f"{prefix}_{m}"]
            std_v = vals.std(ddof=1) if len(vals) > 1 else 0.0
            row[m] = f"{vals.mean():.4f} ± {std_v:.4f}"
        if prefix == "best":
            row["Best Round"] = f"{gdf['Best Round'].mean():.1f}"
            row["Comm at Best (MB)"] = f"{gdf['best_comm_mb'].mean():.1f}"
        rows.append(row)
    return pd.DataFrame(rows)


if not rec_df.empty:
    final_table = mean_std_table(rec_df, "final")
    best_table = mean_std_table(rec_df, "best")
    final_table.to_csv(os.path.join(SAVE_DIR, f"summary_final_mean_std_{MODE}.csv"), index=False, encoding="utf-8-sig")
    best_table.to_csv(os.path.join(SAVE_DIR, f"summary_best_mean_std_{MODE}.csv"), index=False, encoding="utf-8-sig")

    print("\n=== 1) Final(마지막 라운드) Mean ± Std — validation 기준 ===")
    display(final_table)
    print("\n=== 2) [참고] Best(validation 최고 Balanced Acc 라운드) Mean ± Std ===")
    display(best_table)

# 3) Test 결과 (--final_test로 1회 평가한 *_final_test.json). 마지막 라운드 모델이 주 결과, best-val 모델은 참고.
import json

TEST_METRICS = ["balanced_acc", "macro_f1", "macro_f1_fixed8", "macro_f1_gap", "ece"]
test_records = []
for path in sorted(glob.glob(os.path.join(SAVE_DIR, f"*_{MODE}_seed*_final_test.json"))):
    with open(path, encoding="utf-8") as f:
        res = json.load(f)
    cfg = res["config"]
    if FINAL_SEEDS_FILTER is not None and cfg["seed"] not in FINAL_SEEDS_FILTER:
        continue
    if cfg["method"] == "fedprox" and BEST_MU_FILTER is not None and not np.isclose(cfg["mu"], BEST_MU_FILTER):
        continue
    group = f"{cfg['method']} (mu={cfg['mu']:g})" if cfg["method"] == "fedprox" else cfg["method"]
    for which in ("final", "best_val"):
        m = res.get(which)
        if m:
            test_records.append({"Group": group, "Model": which, "Seed": cfg["seed"], **{k: m[k] for k in TEST_METRICS}})

if test_records:
    test_df = pd.DataFrame(test_records)
    for which, title in (("final", "마지막 라운드 모델 — 주 결과"), ("best_val", "best-val 라운드 모델 — 참고")):
        sub = test_df[test_df["Model"] == which]
        if sub.empty:
            continue
        rows = []
        for grp, gdf in sub.groupby("Group", sort=False):
            row = {"Method": grp, "N_Seeds": len(gdf)}
            for m in TEST_METRICS:
                std_v = gdf[m].std(ddof=1) if len(gdf) > 1 else 0.0
                row[m] = f"{gdf[m].mean():.4f} ± {std_v:.4f}"
            rows.append(row)
        table = pd.DataFrame(rows)
        table.to_csv(os.path.join(SAVE_DIR, f"summary_test_{which}_mean_std_{MODE}.csv"), index=False, encoding="utf-8-sig")
        print(f"\n=== 3) Test Mean ± Std — {title} ===")
        display(table)
else:
    print("\n[Test] *_final_test.json 결과가 아직 없습니다. Phase 4를 --final_test로 실행하면 생성됩니다.")


In [ ]:
# 3) 목표 정확도 도달 라운드 및 누적 통신량 (FedAvg vs FedProx)
# 보고서에는 TARGET_ACC를 사전에 정한 고정값(예: 0.60)으로 지정하는 것을 권장합니다.
# None이면 연합학습 그룹별 평균 곡선 최고치의 95%로 자동 설정합니다.
TARGET_ACC = None

fl_dfs = [d for d in all_round_dfs if d["method"].iloc[0] in ("fedavg", "fedprox")]
if fl_dfs:
    if TARGET_ACC is None:
        fl_combined = pd.concat(fl_dfs, ignore_index=True)
        TARGET_ACC = round(float(fl_combined.groupby(["group", "round"])["balanced_acc"].mean().max() * 0.95), 2)

    rows = []
    for d in fl_dfs:
        reached = d[d["balanced_acc"] >= TARGET_ACC]
        hit = reached.iloc[0] if not reached.empty else None
        rows.append({
            "Method": d["group"].iloc[0],
            "Seed": int(d["seed"].iloc[0]),
            "Reached": hit is not None,
            "Rounds to Target": int(hit["round"]) if hit is not None else np.nan,
            "Comm to Target (MB)": float(hit["cumulative_comm_mb"]) if hit is not None else np.nan,
        })
    comm_df = pd.DataFrame(rows)

    # 방법별 집계 (평균은 목표에 도달한 시드만 대상으로 계산되므로 Reached 비율을 함께 확인)
    comm_agg = comm_df.groupby("Method", sort=False).agg(
        Reached=("Reached", lambda s: f"{int(s.sum())}/{len(s)}"),
        Mean_Rounds_to_Target=("Rounds to Target", "mean"),
        Mean_Comm_to_Target_MB=("Comm to Target (MB)", "mean"),
    ).reset_index()

    print(f"=== 3-a) 목표 정확도(Balanced Acc >= {TARGET_ACC:.2f}) 도달 — 시드별 ===")
    display(comm_df)
    print("=== 3-b) 목표 정확도 도달 — 방법별 집계 ===")
    display(comm_agg.round(1))
    comm_agg.to_csv(os.path.join(SAVE_DIR, f"comm_to_target_{MODE}.csv"), index=False, encoding="utf-8-sig")
else:
    print("연합학습(FedAvg/FedProx) 결과가 아직 없습니다.")

In [ ]:
# 4) 라운드별 다중 시드 평균 실선 + 표준편차(±1 Std) 음영 그래프 및 통신 효율성 그래프
if all_round_dfs:
    combined_df = pd.concat(all_round_dfs, ignore_index=True)
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    metrics_to_plot = [
        ("balanced_acc", "Global Balanced Accuracy (↑)", axes[0, 0]),
        ("macro_f1_gap", "Client Max-Min Macro-F1 Gap / Fairness (↓)", axes[0, 1]),
        ("ece", "Expected Calibration Error - 15 Bins (↓)", axes[1, 0]),
        ("weight_divergence", "Client Weight Divergence L2 Norm", axes[1, 1]),
    ]

    for col, title, ax in metrics_to_plot:
        for grp, gdf in combined_df.groupby("group", sort=False):
            stats = gdf.groupby("round")[col].agg(["mean", "std"]).reset_index()
            rounds = stats["round"].values
            mean_vals = stats["mean"].values
            std_vals = stats["std"].fillna(0.0).values
            line, = ax.plot(rounds, mean_vals, marker="o", linewidth=2, label=f"{grp} (n={gdf['seed'].nunique()})")
            if gdf["seed"].nunique() > 1:
                ax.fill_between(rounds, mean_vals - std_vals, mean_vals + std_vals, color=line.get_color(), alpha=0.2)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Communication Round")
        ax.set_ylabel(col)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend()

    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, f"comparison_curves_mean_std_{MODE}.png")
    plt.savefig(plot_path, dpi=200)
    plt.show()
    print(f"[저장 완료] {plot_path}")

    # FedAvg vs FedProx: 누적 통신량(MB) 대비 Balanced Accuracy
    fl_only = combined_df[combined_df["method"].isin(["fedavg", "fedprox"])]
    if not fl_only.empty:
        plt.figure(figsize=(8, 5))
        for grp, gdf in fl_only.groupby("group", sort=False):
            stats = gdf.groupby("cumulative_comm_mb")["balanced_acc"].agg(["mean", "std"]).reset_index()
            comm_x = stats["cumulative_comm_mb"].values
            mean_y = stats["mean"].values
            std_y = stats["std"].fillna(0.0).values
            line, = plt.plot(comm_x, mean_y, marker="s", linewidth=2, label=grp)
            if gdf["seed"].nunique() > 1:
                plt.fill_between(comm_x, mean_y - std_y, mean_y + std_y, color=line.get_color(), alpha=0.2)
        if TARGET_ACC is not None:
            plt.axhline(TARGET_ACC, color="gray", linestyle=":", label=f"Target = {TARGET_ACC:.2f}")
        plt.title("Communication Efficiency: Balanced Accuracy vs. Cumulative Comm (MB)", fontsize=12, fontweight="bold")
        plt.xlabel("Cumulative Communication Volume (MB)")
        plt.ylabel("Global Balanced Accuracy (↑)")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        comm_plot_path = os.path.join(SAVE_DIR, f"comm_efficiency_{MODE}.png")
        plt.savefig(comm_plot_path, dpi=200)
        plt.show()
        print(f"[저장 완료] {comm_plot_path}")
else:
    print("시각화할 CSV 데이터가 아직 없습니다. Phase 3~4를 먼저 실행하세요.")